# 최종 제출 파이프라인 (재현용) — 풍부한 피처셋 CatBoost + ECDF 선형회귀 스태킹

## 결과
- **OOF AUC: 0.74098**
- **실제 제출(Public): 0.7421890122**

## 이 노트북의 범위
처음부터 끝까지 실행하면 위 결과가 **그대로 재현**됩니다. 탐색 과정에서 시도했던 다른 실험
(클리핑 유무 비교, Ridge/Greedy 메타러너 비교, Pseudo-labeling 등)은 이 결과를 만드는 데
사용되지 않았으므로 포함하지 않았습니다 (별도 노트북 참고).

## 전체 흐름
1. 데이터 로드 + 풍부한 피처 엔지니어링 (train/test 동일하게 적용)
2. 새 모델 3종 학습: `donor_cat`(CatBoost, 7시드 배깅), `alpha_cat`(CatBoost+클래스재가중),
   `lgbm_spw2_richfeat`(LightGBM+scale_pos_weight)
3. 기존에 확보해둔 모델 9종의 OOF/test 예측을 `blend_cache`에서 로드 (정확한 12개 후보 목록 고정)
4. ECDF 랭크변환 + 선형회귀 메타러너로 12개 모델을 스태킹 (fold-safe, 데이터 누수 없음)
5. 최종 test 예측 생성 + 제출 파일 저장

## 대회 규칙 준수
- 모든 범주형 인코딩은 **train 데이터에만 fit**, test의 새 값은 sentinel로 안전하게 처리
- 결측치는 고정 상수(-1/sentinel)만 사용, test 통계 미사용
- ECDF 메타러너는 **각 fold의 train 부분에서만** 분포를 적합 (test/val 데이터 누수 없음)

## ⚠️ 실행 전 준비
이 노트북과 같은 위치에서 `collect_files_for_bundle.py`를 먼저 실행해서
`final_submission_bundle/` 폴더에 필요한 파일을 모아두세요. 그 폴더만 있으면
(다른 노트북이나 팀원 폴더를 따로 참조하지 않고) 이 노트북 단독으로 실행됩니다.

## 예상 실행 시간
- CatBoost `donor_cat`(7시드 × 5-fold): 약 80~90분
- CatBoost `alpha_cat`(1시드 × 5-fold): 약 13~14분
- LightGBM `lgbm_spw2_richfeat`(1시드 × 5-fold): 약 3분
- 나머지(스캔, 스태킹, 저장): 1분 이내
- **합계 약 1시간 40분~2시간**


In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings("ignore")


## 0. 경로 / 설정

In [2]:
# ★ 재현성을 위해 모든 입력 파일을 하나의 폴더(BUNDLE_DIR)에서만 읽습니다.
# collect_files_for_bundle.py를 먼저 실행해서 필요한 파일을 이 폴더에 전부 모아두세요.
# 노트북을 다른 위치로 옮기더라도, BUNDLE_DIR 폴더만 같이 옮기면 그대로 작동합니다.
BUNDLE_DIR = Path("./final_submission_bundle")
DATA_DIR = BUNDLE_DIR      # train.csv, test.csv
BLEND_DIR = BUNDLE_DIR     # 모델별 OOF/test 예측 npy 캐시
SUBMIT_DIR = BUNDLE_DIR    # 최종 제출 파일도 같은 폴더에 저장
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

assert BUNDLE_DIR.exists(), (
    f"{BUNDLE_DIR} 폴더가 없습니다 — collect_files_for_bundle.py를 먼저 실행하세요."
)
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]   # 정보량 없는 컬럼 (EDA로 확인)
N_SPLITS = 5

# --- 풍부한 피처 엔지니어링에 쓰이는 상수 ---

# 배아/난자 관련 수치 컬럼: DI 시술이면 전부 구조적으로 결측됨 (Keep-NaN 정책 적용 대상)
EMBRYO_COUNT_COLS = ["총 생성 배아 수", "미세주입된 난자 수", "미세주입에서 생성된 배아 수", "이식된 배아 수",
    "미세주입 배아 이식 수", "저장된 배아 수", "미세주입 후 저장된 배아 수", "해동된 배아 수",
    "해동 난자 수", "수집된 신선 난자 수", "저장된 신선 난자 수", "혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수", "기증자 정자와 혼합된 난자 수"]
EMBRYO_BINARY_COLS = ["단일 배아 이식 여부", "착상 전 유전 진단 사용 여부", "동결 배아 사용 여부",
    "신선 배아 사용 여부", "기증 배아 사용 여부", "대리모 여부"]

# 극단값(이상치) 클리핑 상한 — 분포 꼬리의 노이즈를 줄이기 위함. 원래 초과했는지는 _high_flag로 별도 보존
CLIP_RULES = {"총 생성 배아 수": 40, "수집된 신선 난자 수": 40, "미세주입된 난자 수": 45,
    "혼합된 난자 수": 40, "저장된 배아 수": 30, "배아 이식 경과일": 7, "난자 혼합 경과일": 7,
    "배아 해동 경과일": 2, "난자 해동 경과일": 1}

# 비율(ratio) 피처: (분자 컬럼, 분모 컬럼, 새 컬럼명) — 분모가 0/결측이면 0이 아니라 진짜 NaN으로 처리
RATIO_SPECS = [
    ("총 생성 배아 수", "혼합된 난자 수", "fertilization_rate"),
    ("미세주입에서 생성된 배아 수", "미세주입된 난자 수", "icsi_fertilization_rate"),
    ("이식된 배아 수", "총 생성 배아 수", "embryo_utilization_rate"),
    ("저장된 배아 수", "총 생성 배아 수", "embryo_freezing_rate"),
    ("혼합된 난자 수", "수집된 신선 난자 수", "oocyte_utilization_rate"),
    ("이식된 배아 수", "해동된 배아 수", "thawed_embryo_transfer_ratio"),
]


## 1. 풍부한 피처 엔지니어링

기존 베이스라인은 결측치를 일괄 `-1`로 채웠지만, 여기서는 다음을 추가로 적용:
- **Keep-NaN**: DI 시술의 구조적 결측(배아/난자 컬럼)을 `-1`로 채우지 않고 진짜 NaN으로 유지.
  대신 `_missing`/`_not_applicable_DI` 플래그로 "왜 결측인지"를 명시적으로 표시.
  → CatBoost/LightGBM이 결측치 분기 방향을 학습 시점에 직접 찾는 네이티브 메커니즘을 활용.
- **클리핑**: 극단값 상한 + `_high_flag` (원래 초과했었는지 보존)
- **안전한 비율 피처**: 분모가 0/결측이면 NaN + `_available` 플래그로 "계산 불가능"과 "비율이 0"을 구분
- **실효 모성 나이**: 기증 난자 사용 시 환자 본인 나이 대신 기증자 나이로 대체한 연속형 피처
  (난자 출처×나이 상호작용 EDA에서 발견한 "기증 난자는 나이와 무관하게 안정적" 패턴을 반영)
- **시술유형 토큰화**: `특정 시술 유형` 텍스트에서 하위 토큰을 binary 플래그로 분리
- **네이티브 categorical**: `pd.Categorical` dtype 사용. **카테고리 목록은 train에서만 정의**하고
  (대회 규칙 준수: test 데이터를 인코딩 구조 결정에 사용하지 않음), test에만 있는 새 값이나
  원래 결측이었던 값은 모두 고정 sentinel 문자열("정보없음")로 안전하게 처리


In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw_full = pd.read_csv(TEST_PATH)
test_ids = test_raw_full["ID"]
y = train_raw[TARGET_COL].values.astype(int)


def safe_ratio(df, numerator, denominator, new_col):
    """분모가 0이거나 결측이면 0이 아니라 진짜 NaN을 대입하고, 계산 가능 여부를 별도 플래그로 보존."""
    df = df.copy()
    can_compute = df[numerator].notna() & df[denominator].notna() & (df[denominator] > 0)
    df[f"{new_col}_available"] = can_compute.astype(int)
    df[new_col] = np.where(can_compute, df[numerator] / df[denominator], np.nan)
    return df


def build_full_features(df):
    """train/test에 동일하게 적용하는 피처 엔지니어링 함수. (대회 규칙 준수: train/test를 각각
    독립적으로 처리하며, 이 함수 자체는 어떤 통계도 외부에서 들여오지 않음 — 행 단위 결정론적 변환만 수행)"""
    df = df.copy()
    df = df.drop(columns=[c for c in DEAD_COLS if c in df.columns])
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)

    # --- Keep-NaN: 결측 자체를 채우지 않고, "결측 여부"와 "DI라서 구조적으로 결측인지"를 플래그로 보존 ---
    embryo_cols_present = [c for c in (EMBRYO_COUNT_COLS + EMBRYO_BINARY_COLS) if c in df.columns]
    for col in embryo_cols_present:
        df[f"{col}_missing"] = df[col].isna().astype(int)
        is_di_missing = (df["시술 유형"] == "DI") & df[col].isna()
        df[f"{col}_not_applicable_DI"] = is_di_missing.astype(int)
        # 주의: 원본 컬럼 df[col] 자체는 그대로 NaN으로 남겨둠 (값으로 채우지 않음)

    # --- 아웃라이어 클리핑 ---
    for col, upper in CLIP_RULES.items():
        if col in df.columns:
            df[f"{col}_high_flag"] = (df[col] > upper).astype(int)
            df[col] = df[col].clip(upper=upper)

    # --- 배아 이식 시점 임계값 플래그 (EDA: 5일=배반포 이식일 때 성공률 가장 높음) ---
    if "배아 이식 경과일" in df.columns:
        df["transfer_day_0_1_flag"] = df["배아 이식 경과일"].isin([0, 1]).astype(int)
        df["transfer_day_3_flag"] = (df["배아 이식 경과일"] == 3).astype(int)
        df["transfer_day_5_or_more_flag"] = (df["배아 이식 경과일"] >= 5).astype(int)

    # --- 안전한 비율 피처 ---
    for num, den, new in RATIO_SPECS:
        if num in df.columns and den in df.columns:
            df = safe_ratio(df, num, den, new)

    # --- 실효 모성 나이: 기증 난자 사용 시 환자 나이 대신 기증자 나이로 대체 ---
    patient_mid = {"만18-34세": 31, "만35-37세": 36, "만38-39세": 38.5, "만40-42세": 41,
                    "만43-44세": 43.5, "만45-50세": 47.5, "알 수 없음": np.nan}
    donor_mid = {"만20세 이하": 20, "만21-25세": 23, "만26-30세": 28, "만31-35세": 33,
                 "만36-40세": 38, "만41-45세": 43, "알 수 없음": np.nan}
    if "난자 출처" in df.columns and "시술 당시 나이" in df.columns:
        df["patient_age_mid"] = df["시술 당시 나이"].map(patient_mid)
        donor_age_mid = (df["난자 기증자 나이"].map(donor_mid)
                         if "난자 기증자 나이" in df.columns else pd.Series(np.nan, index=df.index))
        donor_known = (df["난자 출처"] == "기증 제공") & donor_age_mid.notna()
        df["effective_maternal_age_mid"] = df["patient_age_mid"]
        df.loc[donor_known, "effective_maternal_age_mid"] = donor_age_mid[donor_known]
        df["donor_rejuvenation_gap"] = 0.0
        df.loc[donor_known, "donor_rejuvenation_gap"] = (
            df.loc[donor_known, "patient_age_mid"] - donor_age_mid[donor_known]
        )

    # --- 시술유형 텍스트 토큰화 ---
    if "특정 시술 유형" in df.columns:
        s = df["특정 시술 유형"].astype(str)
        for token in ["IVF", "ICSI", "IUI", "ICI", "GIFT", "FER", "Generic DI", "IVI", "BLASTOCYST", "AH"]:
            safe = token.lower().replace(" ", "_")
            df[f"spec_has_{safe}"] = s.str.contains(token, regex=False, na=False).astype(int)

    # --- 인터랙션 카테고리 (나이 x 난자출처) ---
    if {"시술 당시 나이", "난자 출처"}.issubset(df.columns):
        df["age_oocyte_source"] = df["시술 당시 나이"].astype(str) + "_" + df["난자 출처"].astype(str)

    return df


train_feat = build_full_features(train_raw.drop(columns=["ID"]))
test_feat = build_full_features(test_raw_full.drop(columns=["ID"]))

X_train_full = train_feat.drop(columns=[TARGET_COL])
X_test_full = test_feat.copy()

# ★ 대회 규칙 준수: 카테고리 목록은 train에서만 정의. test의 결측/신규값은 모두 같은 sentinel로 흡수
#   (CatBoost는 범주형 컬럼에 NaN을 직접 못 받으므로, 숫자형 Keep-NaN과는 별도로 처리)
SENTINEL = "정보없음"
obj_cols = X_train_full.select_dtypes(include=["object"]).columns.tolist()
for col in obj_cols:
    X_train_full[col] = X_train_full[col].fillna(SENTINEL)
    X_test_full[col] = X_test_full[col].fillna(SENTINEL)
    train_categories = sorted(set(X_train_full[col].astype(str).unique()) | {SENTINEL})
    X_train_full[col] = pd.Categorical(X_train_full[col].astype(str), categories=train_categories)
    X_test_full[col] = pd.Categorical(X_test_full[col].astype(str), categories=train_categories)
    X_test_full[col] = X_test_full[col].fillna(SENTINEL)  # train에 없던 test 신규값도 같은 sentinel로
X_test_full = X_test_full[X_train_full.columns]

print(f"풍부한 피처셋 빌드 완료: train {X_train_full.shape}, test {X_test_full.shape}")
print(f"범주형 컬럼 {len(obj_cols)}개, 전체 피처 {X_train_full.shape[1]}개")


풍부한 피처셋 빌드 완료: train (256351, 145), test (90067, 145)
범주형 컬럼 21개, 전체 피처 145개


## 2. CatBoost 모델 2종

- `donor_cat`: 위 피처셋으로 학습한 기본 CatBoost. **시드 7개로 배깅**(멀티시드 풀파이프라인 배깅 —
  프로젝트 전체에서 유일하게 확실히 검증된 개선 레버, 시드별 표준편차 0.00011 직접 실측)하여
  단일 시드 대비 안정성을 높임. 이 노트북에서 만드는 모델 중 **단독 성능 최고**.
- `alpha_cat`: 동일 피처셋 + `class_weights=[0.6, 0.4]`로 약하게 재가중. 단독 성능은 `donor_cat`보다
  낮지만, 스태킹 단계에서 다른 시각을 더하는 재료로 포함.


In [4]:
def run_catboost(X, y, X_test, seed, class_weights=None, n_splits=N_SPLITS):
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    cat_idx = [X.columns.get_loc(c) for c in cat_cols]
    params = dict(loss_function="Logloss", eval_metric="AUC", iterations=1500, learning_rate=0.03,
                  depth=6, l2_leaf_reg=5, random_seed=seed, early_stopping_rounds=100,
                  allow_writing_files=False, verbose=False)
    if class_weights is not None:
        params["class_weights"] = class_weights

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    for tr_idx, val_idx in skf.split(X, y):
        train_pool = Pool(X.iloc[tr_idx], y[tr_idx], cat_features=cat_idx)
        valid_pool = Pool(X.iloc[val_idx], y[val_idx], cat_features=cat_idx)
        model = CatBoostClassifier(**params)
        model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
        oof[val_idx] = model.predict_proba(valid_pool)[:, 1]
        test_pred += model.predict_proba(Pool(X_test, cat_features=cat_idx))[:, 1] / n_splits
    return oof, test_pred


# --- donor_cat: 7개 시드로 풀파이프라인 배깅 ---
print("=== donor_cat (7시드 평균) ===")
DONOR_SEEDS = [42, 2024, 777, 1234, 5678, 9012, 3456]
donor_oofs, donor_tests = [], []
start = time.time()
for seed in DONOR_SEEDS:
    oof_s, test_s = run_catboost(X_train_full, y, X_test_full, seed=seed)
    donor_oofs.append(oof_s)
    donor_tests.append(test_s)
    print(f"  seed={seed}: AUC={roc_auc_score(y, oof_s):.5f}  ({time.time()-start:.0f}s 누적)")

oof_donor_cat = np.mean(donor_oofs, axis=0)
test_donor_cat = np.mean(donor_tests, axis=0)
print(f"donor_cat(7시드 평균) OOF AUC: {roc_auc_score(y, oof_donor_cat):.5f}")

np.save(BLEND_DIR / "oof_donor_cat.npy", oof_donor_cat)
np.save(BLEND_DIR / "test_donor_cat.npy", test_donor_cat)


=== donor_cat (7시드 평균) ===
  seed=42: AUC=0.74047  (797s 누적)
  seed=2024: AUC=0.74048  (1515s 누적)
  seed=777: AUC=0.74047  (2280s 누적)
  seed=1234: AUC=0.74039  (2984s 누적)
  seed=5678: AUC=0.74042  (3713s 누적)
  seed=9012: AUC=0.74031  (4367s 누적)
  seed=3456: AUC=0.74049  (5205s 누적)
donor_cat(7시드 평균) OOF AUC: 0.74076


In [5]:
# --- alpha_cat: class_weights로 약한 재가중 (스태킹 재료용, 단독 성능은 donor_cat보다 낮음) ---
print("=== alpha_cat (class_weights=[0.6, 0.4]) ===")
start = time.time()
oof_alpha, test_alpha = run_catboost(X_train_full, y, X_test_full, seed=42, class_weights=[0.60, 0.40])
print(f"alpha_cat OOF AUC: {roc_auc_score(y, oof_alpha):.5f}  ({time.time()-start:.0f}s)")

np.save(BLEND_DIR / "oof_alpha_cat.npy", oof_alpha)
np.save(BLEND_DIR / "test_alpha_cat.npy", test_alpha)


=== alpha_cat (class_weights=[0.6, 0.4]) ===
alpha_cat OOF AUC: 0.74033  (823s)


## 3. LightGBM + 풍부한 피처셋 + scale_pos_weight=2.0 (스태킹 재료용)

In [6]:
print("=== lgbm_spw2_richfeat ===")
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
oof_spw = np.zeros(len(y))
test_spw = np.zeros(len(X_test_full))
params_spw = dict(objective="binary", n_estimators=3000, learning_rate=0.018, num_leaves=63,
                   min_child_samples=60, subsample=0.85, colsample_bytree=0.85,
                   reg_alpha=0.05, reg_lambda=2.0, scale_pos_weight=2.0,
                   random_state=42, verbose=-1, n_jobs=-1)

start = time.time()
for tr_idx, val_idx in skf.split(X_train_full, y):
    model = LGBMClassifier(**params_spw)
    model.fit(X_train_full.iloc[tr_idx], y[tr_idx])
    oof_spw[val_idx] = model.predict_proba(X_train_full.iloc[val_idx])[:, 1]
    test_spw += model.predict_proba(X_test_full)[:, 1] / N_SPLITS
print(f"lgbm_spw2_richfeat OOF AUC: {roc_auc_score(y, oof_spw):.5f}  ({time.time()-start:.0f}s)")

np.save(BLEND_DIR / "oof_lgbm_spw2_richfeat.npy", oof_spw)
np.save(BLEND_DIR / "test_lgbm_spw2_richfeat.npy", test_spw)


=== lgbm_spw2_richfeat ===
lgbm_spw2_richfeat OOF AUC: 0.73297  (189s)


## 4. 최종 스태킹에 사용할 12개 모델 후보 로드

**재현성을 위해 후보 목록을 명시적으로 고정합니다.** (이후 다른 실험에서 `blend_cache`에 추가된
모델들 — pseudo-labeling, xgb_richfeat 등 — 이 있어도 이 노트북 결과에는 영향을 주지 않도록,
자동 스캔이 아니라 정확히 이 12개 이름을 지정해서 불러옵니다.)


In [7]:
# 0.74098(실제 0.7421890122)을 만든 정확한 12개 후보 — 이름과 npy 파일명 매핑
FINAL_12_CANDIDATES = {
    "donor_cat":          ("oof_donor_cat.npy", "test_donor_cat.npy"),
    "v5_ensemble":        (None, None),  # 아래에서 별도 처리 (팀원 파일)
    "10seed_bagged":      ("oof_10seed_bagged.npy", "test_lgbm_bagged.npy"),
    "alpha_cat":          ("oof_alpha_cat.npy", "test_alpha_cat.npy"),
    "xgboost_bagged":     ("oof_xgboost_bagged.npy", "test_xgboost_bagged.npy"),
    "catboost_bagged":    ("oof_catboost_bagged.npy", "test_catboost_bagged.npy"),
    "feature_subspace":   ("oof_feature_subspace.npy", "test_feature_subspace.npy"),
    "xgb_rankpairwise":   ("xgb_rankpairwise_oof.npy", "xgb_rankpairwise_test.npy"),
    "lgbm_lambdarank":    ("lgbm_lambdarank_oof.npy", "lgbm_lambdarank_test.npy"),
    "lgbm_keepnan":       ("oof_lgbm_keepnan.npy", "test_lgbm_keepnan.npy"),
    "mlp":                ("oof_mlp.npy", "test_mlp.npy"),
    "lgbm_spw2_richfeat": ("oof_lgbm_spw2_richfeat.npy", "test_lgbm_spw2_richfeat.npy"),
}

models = {}
for name, (oof_file, test_file) in FINAL_12_CANDIDATES.items():
    if name == "v5_ensemble":
        oof_arr = np.load(BUNDLE_DIR / "v5_ensemble_oof.npy")
        test_arr = pd.read_csv(BUNDLE_DIR / "submission_v5_imputed.csv")["probability"].values
    else:
        oof_arr = np.load(BLEND_DIR / oof_file)
        test_arr = np.load(BLEND_DIR / test_file)
    assert len(oof_arr) == len(y), f"{name}: OOF 길이 불일치"
    models[name] = {"oof": oof_arr, "test": test_arr}

all_names = list(models.keys())
assert len(all_names) == 12, f"후보가 12개가 아닙니다: {len(all_names)}개"

print(f"{'모델':<22} | {'OOF AUC':>8}")
print("-" * 36)
for n in all_names:
    print(f"{n:<22} | {roc_auc_score(y, models[n]['oof']):>8.5f}")


모델                     |  OOF AUC
------------------------------------
donor_cat              |  0.74076
v5_ensemble            |  0.74043
10seed_bagged          |  0.74036
alpha_cat              |  0.74033
xgboost_bagged         |  0.74010
catboost_bagged        |  0.74005
feature_subspace       |  0.73973
xgb_rankpairwise       |  0.73939
lgbm_lambdarank        |  0.73748
lgbm_keepnan           |  0.73648
mlp                    |  0.73313
lgbm_spw2_richfeat     |  0.73297


## 5. ECDF 랭크변환 + 선형회귀 메타러너

기존 가중평균(가중치 0~1, 합=1 제약) 대신, **음수 계수를 허용하는 선형회귀**로 모델들을 결합해서
모델간 공유된 노이즈를 상쇄할 수 있게 했습니다.

**fold-safe 설계**: 각 fold의 train 부분에서만 ECDF(경험적 누적분포함수)를 적합하여 모든 모델의
예측을 백분위 스케일로 정규화하고, 그 위에서 선형회귀를 학습 → 평가 fold 정보가 학습 단계에
전혀 들어가지 않음(데이터 누수 없음).


In [8]:
class ECDFReference:
    """주어진 분포(ref)로 fit. transform()은 그 분포 기준 백분위(평균순위/n, 동점 보정)를 반환."""
    def __init__(self, ref):
        self.sorted = np.sort(np.asarray(ref, dtype=float))
        self.n = max(len(self.sorted), 1)

    def transform(self, x):
        x = np.asarray(x, dtype=float)
        left = np.searchsorted(self.sorted, x, side="left")
        right = np.searchsorted(self.sorted, x, side="right")
        return (left + right) / 2.0 / self.n


def make_ecdf_linear_stack(names, models, y, n_splits=N_SPLITS, seed=42):
    oof_matrix = np.column_stack([models[n]["oof"] for n in names])
    test_matrix = np.column_stack([models[n]["test"] for n in names])

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_pred = np.zeros(len(y))
    test_pred = np.zeros(test_matrix.shape[0])
    ncol = len(names)

    for tr, va in skf.split(oof_matrix, y):
        Xtr = np.empty((len(tr), ncol))
        Xva = np.empty((len(va), ncol))
        Xte = np.empty((test_matrix.shape[0], ncol))
        for c in range(ncol):
            # ★ ECDF는 fold의 train 부분(tr)으로만 fit — val(va)/test 정보는 절대 사용하지 않음
            ref = ECDFReference(oof_matrix[tr, c])
            Xtr[:, c] = ref.transform(oof_matrix[tr, c])
            Xva[:, c] = ref.transform(oof_matrix[va, c])
            Xte[:, c] = ref.transform(test_matrix[:, c])

        model = make_pipeline(StandardScaler(), LinearRegression())
        model.fit(Xtr, y[tr])
        oof_pred[va] = np.clip(model.predict(Xva), 0, 1)
        test_pred += np.clip(model.predict(Xte), 0, 1) / n_splits

    return oof_pred, test_pred, roc_auc_score(y, oof_pred)


## 6. 최종 스택 생성 + 검증

In [9]:
final_oof, final_test, final_auc = make_ecdf_linear_stack(all_names, models, y)

print(f"최종 OOF AUC: {final_auc:.5f}")
print(f"(기대값: 0.74098 — 차이가 크면 후보 모델이나 데이터가 달라진 것이니 확인 필요)")

assert abs(final_auc - 0.74098) < 0.0005, (
    f"OOF AUC({final_auc:.5f})가 기대값(0.74098)과 크게 다릅니다 — "
    "모델 재현에 문제가 있을 수 있으니 위 12개 후보의 개별 OOF를 다시 확인하세요."
)
print("\n✅ 재현성 확인 완료")


최종 OOF AUC: 0.74098
(기대값: 0.74098 — 차이가 크면 후보 모델이나 데이터가 달라진 것이니 확인 필요)

✅ 재현성 확인 완료


## 7. 최종 제출 파일 생성

In [10]:
SUBMIT_DIR.mkdir(exist_ok=True)
submission = pd.DataFrame({"ID": test_ids, "probability": final_test})
out_path = SUBMIT_DIR / f"2차_ecdf_stack_FINAL_local{final_auc:.5f}.csv"
submission.to_csv(out_path, index=False)

print(f"저장 완료: {out_path}")
print(f"이 파일이 Public 리더보드 기준 0.7421890122를 받은 제출 파일과 동일해야 합니다.")


저장 완료: final_submission_bundle/2차_ecdf_stack_FINAL_local0.74098.csv
이 파일이 Public 리더보드 기준 0.7421890122를 받은 제출 파일과 동일해야 합니다.
